In [2]:
import os
import numpy as np
import sympy as sp
import torch

from SymbolicMathematics.src.envs import set_cover_utils
from src.utils import AttrDict
from src.envs import build_env
from src.model import build_modules

from src.utils import to_cuda
from src.envs.sympy_utils import simplify
from src.envs.set_cover_utils import validate_set_cover_solution

from src.envs import char_sp
from src.envs.char_sp import CharSPEnvironment
from itertools import combinations

import json
import matplotlib.pyplot as plt

## Build environment / Reload model

In [3]:
# trained model, e.g. "wget https://dl.fbaipublicfiles.com/SymbolicMathematics/models/fwd_bwd.pth"
#model_path = './fwd_bwd.pth'
model_path1 = './Model1/first_train/2spv1sq7hj/checkpoint.pth'
assert os.path.isfile(model_path1)

In [4]:
params = params = AttrDict({

    # environment parameters
    'env_name': 'char_sp',
    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,
    'n_variables': 1,
    'n_coefficients': 0,
    'leaf_probs': '0.75,0,0.25,0',
    'max_len': 512,
    'max_int': 5,
    'max_ops': 15,
    'max_ops_G': 15,
    'clean_prefix_expr': True,
    'rewrite_functions': '',
    #'tasks': 'prim_fwd',
    'tasks': 'set_cover',
    'operators': 'add:10,sub:3,mul:10,div:5,sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,ln:4,exp:4,sin:4,cos:4,tan:4,asin:1,acos:1,atan:1,sinh:1,cosh:1,tanh:1,asinh:1,acosh:1,atanh:1',

    # model parameters
    'cpu': True,
    'emb_dim': 128,
    'n_enc_layers': 2, #it was 6
    'n_dec_layers': 2, #it was 6
    'n_heads': 2, #it was 8
    'dropout': 0,
    'attention_dropout': 0,
    'sinusoidal_embeddings': False,
    'share_inout_emb': True,
    'reload_model': model_path1,

})

In [5]:
env = build_env(params)
#x = env.local_dict['x']

In [6]:
modules = build_modules(env, params)
encoder = modules['encoder']
decoder = modules['decoder']

In [7]:
def convert_token_sequence_to_list_of_lists(token_sequence):
    list_of_lists = []  # This will store the final list of lists
    current_list = []  # This will store the current list being constructed
    depth = 0  # Variable to keep track of the depth of nesting
    inner_list = []  # Initialize inner_list

    if len(token_sequence) == 0:
        return []

    for token in token_sequence:  # Iterate over each token in the token sequence
        if token == '[':  # If an opening bracket is encountered
            if depth > 0:
                inner_list = []  # Start a new inner list
            depth += 1  # Increment the depth of nesting
        elif token == ']':  # If a closing bracket is encountered
            depth -= 1  # Decrement the depth of nesting
            if depth == 0:
                if inner_list:  # Check if the inner list is not empty
                    current_list.append(inner_list)  # Append the inner list to the current list
                list_of_lists.append(current_list)  # Append the constructed list to the final result
                current_list = []  # Reset the current list
            else:
                current_list.append(inner_list)  # Append the inner list to the current list
                inner_list = []  # Start a new inner list
        elif token.isdigit():  # If the token is an integer
            inner_list.append(int(token))  # Convert token to integer and add to the inner list

    if len(list_of_lists) == 0:
        return []
    return list_of_lists[0]  # Return the final list of lists

In [74]:
def validate_set_cover_solution(problem_set, solution_set, true_output):
    valid_solution_count = 0
    error_counts = {
        "Wrong Format": 0,
        "Subset not in problem set": 0,
        "Solution not covering problem set": 0,
        "Solution is not optimal": 0
    }
    coverage_percentage = 0

    # Check if the solution set is empty
    if not solution_set:
        error_counts["Wrong Format"] += 1
        return valid_solution_count, error_counts, coverage_percentage


    # Check if any subset in the solution set is empty
    for j, subset in enumerate(solution_set):
        if not subset:
            error_counts["Wrong Format"] += 1
            return valid_solution_count, error_counts, coverage_percentage

    # Check if all subsets in the solution set exist in the problem set
        problem_set_set = [frozenset(subset) for subset in problem_set]
        for subset in solution_set:
            if frozenset(subset) not in problem_set_set:
                error_counts["Subset not in problem set"] += 1
                return valid_solution_count, error_counts, coverage_percentage

    
        # Check if the solution set covers the entire problem set
    covered_tokens = set().union(*solution_set)
    problem_tokens = set(token for sublist in problem_set for token in sublist)
    if covered_tokens != problem_tokens:
        coverage_percentage = round(len(covered_tokens) / len(problem_tokens) * 100, 2)
        error_counts["Solution not covering problem set"] += 1

    else:
        if len(solution_set) > len(true_output):
            error_counts["Solution is not optimal"] += 1
        else:
            valid_solution_count += 1

    return valid_solution_count, error_counts, coverage_percentage

In [67]:
def list_to_tokens(lst):
    return [str(c) for c in str(lst) if str(c) != ' ']

In [75]:
problem_set_instance = [[1, 2, 3, 6], [0, 2], [0, 1], [0, 1, 3], [0, 2, 3], [0, 5], [0, 1, 3, 4]]
true_set_instance = [[1, 2, 3]]

In [76]:
problem_instance_tokens = list_to_tokens(problem_set_instance)
true_instance_tokens = list_to_tokens(true_set_instance)
print(problem_instance_tokens)
print(true_instance_tokens)

['[', '[', '1', ',', '2', ',', '3', ',', '6', ']', ',', '[', '0', ',', '2', ']', ',', '[', '0', ',', '1', ']', ',', '[', '0', ',', '1', ',', '3', ']', ',', '[', '0', ',', '2', ',', '3', ']', ',', '[', '0', ',', '5', ']', ',', '[', '0', ',', '1', ',', '3', ',', '4', ']', ']']
['[', '[', '1', ',', '2', ',', '3', ']', ']']


### Encode input

In [77]:
#x1_prefix = env.clean_prefix(['sub', 'derivative', 'f', 'x', 'x'] + f_prefix)
x1_prefix = problem_instance_tokens
x1 = torch.LongTensor(
    [env.eos_index] +
    [env.word2id[w] for w in x1_prefix] +
    [env.eos_index]
).view(-1, 1)
len1 = torch.LongTensor([len(x1)])
#x1, len1 = to_cuda(x1, len1)

with torch.no_grad():
    encoded = encoder('fwd', x=x1, lengths=len1, causal=False).transpose(0, 1)

### Decode with beam search

In [78]:
beam_size = 1
with torch.no_grad():
    _, _, beam = decoder.generate_beam(encoded, len1, beam_size=beam_size, length_penalty=1.0, early_stopping=1, max_len=200)
    assert len(beam) == 1
hypotheses = beam[0].hyp
assert len(hypotheses) == beam_size

### Print results

In [79]:
#print(f"Input function f: {f}")
#print(f"Reference function F: {F}")
#print("")
print("Problem:",problem_set_instance)
for score, sent in sorted(hypotheses, key=lambda x: x[0], reverse=True):

    # parse decoded hypothesis
    ids = sent[1:].tolist()                  # decoded token IDs
    tok = [env.id2word[wid] for wid in ids]  # convert to prefix

    try:
        hyp = convert_token_sequence_to_list_of_lists(tok)

    except:
        res = "INVALID PREFIX EXPRESSION"
        hyp = tok
    # print result
    print("%.5f  %s" % (score, hyp))

Problem: [[1, 2, 3, 6], [0, 2], [0, 1], [0, 1, 3], [0, 2, 3], [0, 5], [0, 1, 3, 4]]
-0.08437  [[1, 2, 3, 5], [0, 1, 3, 4]]


In [83]:
#Just checking validation:
problem_set_instance = [[1, 2, 3], [0, 2], [6, 0]]
hyp = [[1, 2, 3], [0, 6]]
true_set_instance = [[1, 2, 3], [0, 6]]

In [84]:
# Validity 
valid_solution_count1, error_counts1, coverage_percentagecheck1 = validate_set_cover_solution(problem_set_instance, hyp, true_set_instance)
print("Problem:", problem_set_instance)
print("decoded:", hyp)
print("true:", true_set_instance)
print(valid_solution_count1, error_counts1, coverage_percentagecheck1)

Problem: [[1, 2, 3], [0, 2], [6, 0]]
decoded: [[1, 2, 3], [0, 6]]
true: [[1, 2, 3], [0, 6]]
1 {'Wrong Format': 0, 'Subset not in problem set': 0, 'Solution not covering problem set': 0, 'Solution is not optimal': 0} 0


In [82]:
def convert_token_sequence_to_list_of_lists(token_sequence):
    list_of_lists = []  # This will store the final list of lists
    current_list = []  # This will store the current list being constructed
    depth = 0  # Variable to keep track of the depth of nesting
    inner_list = []  # Initialize inner_list

    if len(token_sequence) == 0:
        return []

    for token in token_sequence:  # Iterate over each token in the token sequence
        if token == '[':  # If an opening bracket is encountered
            if depth > 0:
                inner_list = []  # Start a new inner list
            depth += 1  # Increment the depth of nesting
        elif token == ']':  # If a closing bracket is encountered
            depth -= 1  # Decrement the depth of nesting
            if depth == 0:
                if inner_list:  # Check if the inner list is not empty
                    current_list.append(inner_list)  # Append the inner list to the current list
                list_of_lists.append(current_list)  # Append the constructed list to the final result
                current_list = []  # Reset the current list
            else:
                current_list.append(inner_list)  # Append the inner list to the current list
                inner_list = []  # Start a new inner list
        elif token.isdigit():  # If the token is an integer
            inner_list.append(int(token))  # Convert token to integer and add to the inner list

    if len(list_of_lists) == 0:
        return []
    return list_of_lists[0]  # Return the final list of lists

In [64]:
def set_cover_greedy(input_sequence):
        # Initialize variables
        universe = set()
        for subset in input_sequence:
            universe.update(subset)
        
        universe = list(universe)
        covered = set()
        selected_subsets = []

        # Iterate until all elements are covered
        while covered != set(universe):
            best_subset = None
            best_cover = set()
            
            # Find the subset that covers the most uncovered elements
            for subset in input_sequence:
                cover = set(subset) - covered
                if len(cover) > len(best_cover):
                    best_cover = cover
                    best_subset = subset
            
            if best_subset is None:
                break
            
            # Add the best subset to the solution and update covered elements
            selected_subsets.append(best_subset)
            covered.update(best_cover)

        return selected_subsets

In [65]:
approximation_sequence = set_cover_greedy(problem_set_instance)
print("Input Sequence:", problem_set_instance)
print("Approximation Sequence:", approximation_sequence)

Input Sequence: [[1, 2, 3], [0, 2, 3], [0, 1], [0, 1, 3]]
Approximation Sequence: [[1, 2, 3], [0, 2, 3]]
